# FusionUncertaintyNet — Kaggle P100 Training
Runs on P100 (16GB) with frozen ESM2/ProtT5. Checkpoints pushed to HF `bhumika-tewari-282006/fusionuncertaintynet-checkpoints` every epoch.

**Secrets needed in Kaggle > Settings > Secrets:**
- `HF_TOKEN` = hf_xxx... (bhumika)
- `KAGGLE_JSON` optional (already mounted)

Do not commit tokens.

In [ ]:
!nvidia-smi
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
!pip -q install fair-esm transformers==4.41.2 huggingface_hub==0.24.0 biopython scikit-learn
import os, sys
sys.path.append('/kaggle/input/fusionuncertaintynet-repo/backend-heavy')
# clone repo if not present
!if [ ! -d "/kaggle/working/FusionUncertaintyNet" ]; then git clone https://github.com/Anamitra-Sarkar/FusionUncertaintyNet.git /kaggle/working/FusionUncertaintyNet; fi
!ls /kaggle/working/FusionUncertaintyNet

In [ ]:
import os
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    # try kaggle secrets
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        print('Got HF_TOKEN from Kaggle Secrets')
    except Exception as e:
        print('HF_TOKEN not found:', e)
else:
    print('HF_TOKEN from env', HF_TOKEN[:10]+'...')

os.environ['HF_TOKEN'] = HF_TOKEN or ''
# verify GPU
import torch
print('CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_properties(0))

In [ ]:
# prepare manifest — for demo we use synthetic. For real AFdb, download via HF dataset DeepMind/afdb or kaggle dataset.
!mkdir -p /kaggle/working/data
!python /kaggle/working/FusionUncertaintyNet/training/scripts/dataset.py  # if needed
# Build synthetic manifest if not exists
import json, random, os
from pathlib import Path
manifest = "/kaggle/working/data/manifest.jsonl"
if not os.path.exists(manifest):
    print('Building synthetic manifest...')
    # import builder
    import sys; sys.path.append('/kaggle/working/FusionUncertaintyNet/training/scripts')
    from dataset import build_manifest_from_afdb
    build_manifest_from_afdb('', '', manifest, max_items=3000)
    print('Done', manifest)
!wc -l /kaggle/working/data/manifest.jsonl && head -n 2 /kaggle/working/data/manifest.jsonl | cut -c1-200

In [ ]:
# Train — P100 tuned: batch implicit 16, grad_accum 2, fp16 autocast
!cd /kaggle/working/FusionUncertaintyNet && python training/scripts/train.py --manifest /kaggle/working/data/manifest.jsonl --epochs 10 --lr 1e-4 --hf_repo bhumika-tewari-282006/fusionuncertaintynet-checkpoints --out /kaggle/working/checkpoints --synthetic
!ls -R /kaggle/working/checkpoints | head -n 50

In [ ]:
# Evaluate & push final metrics
!python /kaggle/working/FusionUncertaintyNet/training/scripts/evaluate.py
# ensure best is pushed
from huggingface_hub import HfApi
import os
tok = os.environ.get('HF_TOKEN')
if tok:
    api=HfApi()
    try:
        print(api.list_models(author='bhumika-tewari-282006'))
    except Exception as e:
        print(e)
else:
    print('No token')

## Notes
- If OOM on P100, reduce synthetic L max to 150 or set `use_prott5=False` in `extract_all` fallback.
- Real AFdb training: replace manifest with AFdb+PDB aligned lDDT. Ensure UniRef clustering split.
- Every epoch auto-pushes to HF — if Kaggle dies, resume from last checkpoint via `MODEL_PATH`.